# RQ2 生成回答（say）— Colab

對 **192 道壓制題(S2)** 用 Qwen / Gemma(4-bit)生成回答,輸出 `responses_*.jsonl`。
含**續跑**:斷線後重跑會自動略過已完成的。

**用法**:執行階段→變更執行階段類型→**T4 GPU**→儲存;然後由上往下逐格執行。
跑完 Qwen 後,把步驟 3 的 `MODEL` 改成 Gemma,再重跑步驟 3–6 即可。


## 步驟 0：確認 GPU


In [ ]:
!nvidia-smi


## 步驟 1：安裝套件（約 1 分鐘）


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pandas


## 步驟 2：上傳 `rq2_stimuli_FINAL.csv`（只保留 192 道壓制題）


In [ ]:
from google.colab import files
import pandas as pd
up = files.upload()
fn = list(up.keys())[0]
df = pd.read_csv(fn)
df = df[df['evidence_line'] == 'suppression'].reset_index(drop=True)
print('壓制題數:', len(df))
df[['subject','lang','stance_strength','concept_en']].head()


## 步驟 3：設定（選模型；Gemma 要填 HF token）
- 跑 **Qwen**：`MODEL` 用 Qwen,`HF_TOKEN` 留空。
- 跑 **Gemma**：`MODEL` 改成 gemma,並填**你自己的** `HF_TOKEN`(先在網頁按過 Gemma 的 Agree)。


In [ ]:
MODEL = 'Qwen/Qwen2.5-7B-Instruct'      # 換 Gemma: 'google/gemma-3-12b-it'
HF_TOKEN = ''                            # Gemma 才需要填 hf_xxx;Qwen 留空
MAX_NEW_TOKENS = 256


## 步驟 3.5：掛載 Google Drive（強烈建議，防斷線白費）
輸出寫到 Drive,VM 重置也不會丟,續跑能真正接上。


In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/rq2_responses'
os.makedirs(SAVE_DIR, exist_ok=True)
print('輸出會存到:', SAVE_DIR)


## 步驟 4：載入模型（4-bit，約 1–2 分鐘）


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
tokn = HF_TOKEN or None
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL, token=tokn)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto', token=tokn)
model.eval()
print('載入完成:', MODEL)


## 步驟 5：生成（含續跑；T4 上約 30–50 分鐘 / 模型）
斷線就重跑這格,會從沒做完的接著跑。跑的時候**別關分頁**。


In [ ]:
import json, os, time
def generate(prompt):
    text = tok.apply_chat_template([{'role':'user','content':prompt}], tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

OUT = os.path.join(globals().get("SAVE_DIR","."), f"responses_{MODEL.split('/')[-1]}.jsonl")
META = ['pair_id','sent_seq','lang','subject','subject_en','concept_en','stance_strength','evidence_line','text']
done = set()
if os.path.exists(OUT):
    for line in open(OUT, encoding='utf-8'):
        try: done.add(str(json.loads(line)['sent_seq']))
        except: pass
todo = df[~df['sent_seq'].astype(str).isin(done)]
print(f'共{len(df)}｜已完成{len(done)}｜本輪{len(todo)} -> {OUT}')
t0 = time.time()
with open(OUT, 'a', encoding='utf-8') as f:
    for i, (_, r) in enumerate(todo.iterrows(), 1):
        resp = generate(str(r['text']))
        rec = {c: ('' if pd.isna(r[c]) else r[c]) for c in META}
        rec.update(model=MODEL, response=resp)
        f.write(json.dumps(rec, ensure_ascii=False) + '\n'); f.flush()
        if i % 10 == 0 or i == len(todo): print(f'{i}/{len(todo)}  ({time.time()-t0:.0f}s)', flush=True)
print('完成 ✅', OUT)


## 步驟 6：下載結果


In [ ]:
from google.colab import files
files.download(OUT)


## 跑另一個模型
回步驟 3 把 `MODEL` 改成 `google/gemma-3-12b-it`、填 `HF_TOKEN`,再依序跑步驟 3→4→5→6。
兩個模型的輸出檔名不同(`responses_Qwen2.5-7B-Instruct.jsonl` / `responses_gemma-3-12b-it.jsonl`),不會互蓋。
